# Day 21 — 六维共振策略压力测试

**目标**：把 Day 20 的六维共振策略放到五只不同市场环境的股票上，找出策略的边界条件。

五大测试：
1. 五股全景扫描（强熊→强牛）
2. 维度组合胜率排行
3. 市场环境 vs 信号质量
4. 退出方式对比
5. 六维度真实权重

In [ ]:
import pandas as pd
import numpy as np

# ===== 通用函数 =====
def calc_rsi(series, period=14):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = (-delta).clip(lower=0)
    avg_gain = gain.ewm(alpha=1/period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

def calc_all_signals(df):
    """六维共振全信号计算"""
    d = df.copy()
    d['MA5'] = d['收盘'].rolling(5).mean()
    d['MA20'] = d['收盘'].rolling(20).mean()
    d['MA60'] = d['收盘'].rolling(60).mean()
    ema12 = d['收盘'].ewm(span=12, adjust=False).mean()
    ema26 = d['收盘'].ewm(span=26, adjust=False).mean()
    d['DIF'] = ema12 - ema26
    d['DEA'] = d['DIF'].ewm(span=9, adjust=False).mean()
    d['RSI'] = calc_rsi(d['收盘'], 14)
    bb_mid = d['收盘'].rolling(20).mean()
    bb_std = d['收盘'].rolling(20).std()
    d['%B'] = (d['收盘'] - (bb_mid - 2*bb_std)) / (4*bb_std)
    d['量MA20'] = d['成交量'].rolling(20).mean()
    # K线形态
    d['实体'] = abs(d['收盘'] - d['开盘'])
    d['下影'] = d[['开盘','收盘']].min(axis=1) - d['最低']
    d['上影'] = d['最高'] - d[['开盘','收盘']].max(axis=1)
    d['K线长度'] = d['最高'] - d['最低']
    d['锤子'] = ((d['下影']>=2*d['实体']) & (d['上影']<=0.3*d['实体']) & (d['实体']>0) & (d['收盘']<d['收盘'].shift(5))).astype(int)
    d['吞没'] = ((d['收盘']>d['开盘']) & (d['收盘'].shift(1)<d['开盘'].shift(1)) & (d['开盘']<d['收盘'].shift(1)) & (d['收盘']>d['开盘'].shift(1))).astype(int)
    # 六维信号
    d['SIG_MA'] = (d['MA5'] > d['MA20']).astype(int)
    d['SIG_MACD'] = (d['DIF'] > d['DEA']).astype(int)
    d['SIG_RSI'] = ((d['RSI'] > 30) & (d['RSI'] < 50)).astype(int)
    d['SIG_BB'] = (d['%B'] < 0.3).astype(int)
    d['SIG_VOL'] = (d['成交量'] > 1.5 * d['量MA20']).astype(int)
    d['SIG_KLINE'] = ((d['锤子']+d['吞没'])>0).astype(int)
    d['共振分数'] = d['SIG_MA']+d['SIG_MACD']+d['SIG_RSI']+d['SIG_BB']+d['SIG_VOL']+d['SIG_KLINE']
    return d

def backtest_strategy(df, buy_score=3, hold_days=10, stop_loss=-0.05, trend_filter=False):
    """V1/V2策略回测"""
    trades = []
    in_position = False
    for i in range(len(df)):
        if i < 60:
            continue
        if not in_position:
            cond_score = df['共振分数'].iloc[i] >= buy_score
            cond_macd = df['DIF'].iloc[i] > df['DEA'].iloc[i]
            cond_vol = df['SIG_VOL'].iloc[i] == 1
            cond_trend = df['MA20'].iloc[i] > df['MA60'].iloc[i] if trend_filter else True
            if cond_score and cond_macd and cond_vol and cond_trend:
                in_position = True
                buy_idx, buy_price = i, df['收盘'].iloc[i]
        else:
            days_held = i - buy_idx
            ret = df['收盘'].iloc[i] / buy_price - 1
            macd_dead = (df['DIF'].iloc[i] < df['DEA'].iloc[i]) and (df['DIF'].iloc[i-1] >= df['DEA'].iloc[i-1])
            if days_held >= hold_days or ret <= stop_loss or macd_dead:
                exit_type = '到期' if days_held>=hold_days else ('止损' if ret<=stop_loss else '死叉')
                trades.append({'buy_date': df.index[buy_idx], 'sell_date': df.index[i],
                               'return': ret*100, 'days': days_held, 'exit': exit_type,
                               'score': int(df['共振分数'].iloc[buy_idx])})
                in_position = False
    return pd.DataFrame(trades)

def macd_baseline(df, hold_days=10):
    """纯MACD金叉基准"""
    trades = []
    in_position = False
    for i in range(1, len(df)):
        if not in_position:
            if df['DIF'].iloc[i] > df['DEA'].iloc[i] and df['DIF'].iloc[i-1] <= df['DEA'].iloc[i-1]:
                in_position = True
                buy_idx, buy_price = i, df['收盘'].iloc[i]
        else:
            days_held = i - buy_idx
            ret = df['收盘'].iloc[i] / buy_price - 1
            macd_dead = (df['DIF'].iloc[i] < df['DEA'].iloc[i]) and (df['DIF'].iloc[i-1] >= df['DEA'].iloc[i-1])
            if days_held >= hold_days or ret <= -0.05 or macd_dead:
                trades.append({'buy_date': df.index[buy_idx], 'sell_date': df.index[i],
                               'return': ret*100, 'days': days_held})
                in_position = False
    return pd.DataFrame(trades)

# 股票列表
STOCKS = [('茅台','/tmp/mt.csv'), ('招商银行','/tmp/zs.csv'), ('比亚迪','/tmp/byd.csv'),
          ('恒瑞医药','/tmp/恒瑞医药.csv'), ('海康威视','/tmp/海康威视.csv')]

print('✅ 函数加载完成，五只股票就绪')

---
## 测试一：五股全景扫描

用 V1 共振、V2 趋势过滤、MACD 基准三种方法，在五只股票上回测。

In [ ]:
print(f'{"股票":8s} {"区间收益":>8s} {"环境":6s} {"V1笔数":>5s} {"V1胜率":>6s} {"V1累计":>7s} {"MACD笔数":>6s} {"MACD累计":>8s} {"胜者":>10s}')
print('-' * 85)

for name, path in STOCKS:
    df = pd.read_csv(path)
    df['日期'] = pd.to_datetime(df['日期'])
    df = df.set_index('日期').sort_index()
    d = calc_all_signals(df).iloc[60:]

    total_ret = (d['收盘'].iloc[-1] / d['收盘'].iloc[0] - 1) * 100
    ma_pct = (d['MA20'] > d['MA60']).sum() / len(d) * 100

    # 按MA多头占比分类
    if ma_pct > 50: regime = '强牛'
    elif ma_pct > 35: regime = '弱牛'
    elif ma_pct > 25: regime = '弱熊'
    else: regime = '强熊'

    v1 = backtest_strategy(d, trend_filter=False)
    mb = macd_baseline(d)

    def summarize(trades):
        if len(trades) == 0:
            return 0, 'N/A', 0.0
        wr = (trades['return'] > 0).sum() / len(trades) * 100
        tr = trades['return'].sum()
        return len(trades), f'{wr:.0f}%', tr

    n1, wr1, tr1 = summarize(v1)
    nm, wrm, trm = summarize(mb)
    winner = 'V1共振 ✅' if tr1 > trm else ('MACD ✅' if trm > tr1 else '平手')

    print(f'{name:6s} {total_ret:+7.1f}% {regime:6s} {n1:5d} {wr1:>6s} {tr1:+7.2f}% {nm:6d} {trm:+8.2f}% {winner:>10s}')

print('\n结论：熊市共振胜，牛市MACD胜。没有一招通吃。')

---
## 测试二：维度组合胜率排行

不是所有≥3分的组合都一样。哪种三维组合最好？哪种最差？

In [ ]:
from itertools import combinations

DIM_NAMES = ['MA', 'MACD', 'RSI', 'BB', 'VOL', 'KLINE']
all_combos = []

for name, path in STOCKS:
    df = pd.read_csv(path)
    df['日期'] = pd.to_datetime(df['日期'])
    df = df.set_index('日期').sort_index()
    d = calc_all_signals(df).iloc[60:]
    dims = [d['SIG_MA'], d['SIG_MACD'], d['SIG_RSI'], d['SIG_BB'], d['SIG_VOL'], d['SIG_KLINE']]

    for r in range(3, 5):  # 3-4个维度
        for combo in combinations(range(6), r):
            mask = dims[combo[0]] == 1
            for c in combo[1:]:
                mask = mask & (dims[c] == 1)

            idxs = d[mask].index
            wins, total = 0, 0
            for idx_val in idxs:
                try:
                    pos = d.index.get_loc(idx_val)
                    if pos + 10 < len(d):
                        ret = d['收盘'].iloc[pos+10] / d['收盘'].iloc[pos] - 1
                        total += 1
                        wins += 1 if ret > 0 else 0
                except: pass

            if total >= 3:
                all_combos.append({'组合': '+'.join([DIM_NAMES[i] for i in combo]),
                                   '信号数': total, '胜率': wins/total*100, '股票': name})

combo_df = pd.DataFrame(all_combos)
agg = combo_df.groupby('组合').agg(信号数=('信号数','sum'), 平均胜率=('胜率','mean'), 覆盖股数=('股票','nunique'))
agg = agg.sort_values('平均胜率', ascending=False)

print('🏆 胜率最高的维度组合（≥3次信号）：')
print(agg.head(8).to_string())
print(f'\n📉 胜率最低的维度组合：')
print(agg.tail(3).to_string())
print('\n→ 包含RSI或MACD的组合普遍优于包含KLINE的组合')

---
## 测试三：市场环境分档 vs 信号胜率

按过去20日涨跌幅分七档，每档内共振信号的10日胜率。

In [ ]:
all_signals = []
for name, path in STOCKS:
    df = pd.read_csv(path)
    df['日期'] = pd.to_datetime(df['日期'])
    df = df.set_index('日期').sort_index()
    d = calc_all_signals(df).iloc[60:]
    d['past20'] = d['收盘'].pct_change(20) * 100
    d['stock'] = name
    all_signals.append(d)

combined = pd.concat(all_signals)

bins = [(-100,-10,'暴跌(<-10%)'), (-10,-5,'强熊'), (-5,-2,'弱熊'),
        (-2,2,'震荡'), (2,5,'弱牛'), (5,10,'强牛'), (10,100,'暴涨(>+10%)')]

print(f'{"过去20日环境":16s} {"信号数":>5s} {"10日胜率":>8s}')
print('-' * 40)

for lo, hi, label in bins:
    seg = combined[(combined['past20'] > lo) & (combined['past20'] <= hi)]
    mask = seg['共振分数'] >= 3
    idxs = seg[mask].index
    wins, total = 0, 0
    for idx_val in idxs:
        try:
            pos = seg.index.get_loc(idx_val)
            if pos + 10 < len(seg):
                ret = seg['收盘'].iloc[pos+10] / seg['收盘'].iloc[pos] - 1
                total += 1
                wins += 1 if ret > 0 else 0
        except: pass
    wr = f'{wins/total*100:.0f}%' if total > 0 else 'N/A'
    bar = '█' * mask.sum()
    print(f'{label:14s} {mask.sum():5d} {wr:>8s}')

print('\n→ 过去20日涨5-10%（强牛）是共振策略的甜蜜点！')

---
## 测试四：退出方式对比

到期卖出 vs 死叉退出 vs 止损退出，哪种最好？

In [ ]:
all_trades = []
for name, path in STOCKS:
    df = pd.read_csv(path)
    df['日期'] = pd.to_datetime(df['日期'])
    df = df.set_index('日期').sort_index()
    d = calc_all_signals(df)
    trades = backtest_strategy(d, trend_filter=False)
    all_trades.append(trades)

all_t = pd.concat(all_trades)

for ext in ['到期', '死叉', '止损']:
    sub = all_t[all_t['exit'] == ext]
    if len(sub) > 0:
        wr = (sub['return'] > 0).sum() / len(sub) * 100
        print(f'{ext}退出 ({len(sub)}笔): 胜率{wr:.0f}% | 均值{sub["return"].mean():+.2f}% | 持有{sub["days"].mean():.0f}天')

print(f'\n⚠️ 死叉退出全部亏损！建议取消死叉退出，仅保留到期+止损。')

---
## 测试五：六维度真实权重

每个维度单独满足时，10日胜率是多少？在牛市和熊市中有何不同？

In [ ]:
dim_results = []
for name, path in STOCKS:
    df = pd.read_csv(path)
    df['日期'] = pd.to_datetime(df['日期'])
    df = df.set_index('日期').sort_index()
    d = calc_all_signals(df).iloc[60:]
    is_bull = (d['收盘'].iloc[-1] / d['收盘'].iloc[0] - 1) > 0

    for dim_name, col in [('MA','SIG_MA'),('MACD','SIG_MACD'),('RSI','SIG_RSI'),
                          ('BB','SIG_BB'),('VOL','SIG_VOL'),('KLINE','SIG_KLINE')]:
        mask = d[col] == 1
        idxs = d[mask].index
        wins, total = 0, 0
        for idx_val in idxs:
            try:
                pos = d.index.get_loc(idx_val)
                if pos + 10 < len(d):
                    ret = d['收盘'].iloc[pos+10] / d['收盘'].iloc[pos] - 1
                    total += 1
                    wins += 1 if ret > 0 else 0
            except: pass
        if total > 0:
            dim_results.append({'维度': dim_name, '股票': name, '胜率': wins/total*100,
                               '市场': '牛市' if is_bull else '熊市', '信号频率': total/len(d)*100})

dr = pd.DataFrame(dim_results)
print(f'{"维度":6s} {"平均胜率":>7s} {"熊市胜率":>8s} {"牛市胜率":>8s} {"信号频率":>7s} {"建议权重":>8s}')
print('-' * 55)

for dim in ['MA','MACD','RSI','BB','VOL','KLINE']:
    sub = dr[dr['维度'] == dim]
    bear = sub[sub['市场'] == '熊市']
    bull = sub[sub['市场'] == '牛市']
    avg_wr = sub['胜率'].mean()
    bwr = bear['胜率'].mean() if len(bear) > 0 else 0
    buwr = bull['胜率'].mean() if len(bull) > 0 else 0
    freq = sub['信号频率'].mean()
    # 建议权重：胜率×稀缺性（1/频率）的归一化
    weight = avg_wr * (1 / max(freq, 1))
    stars = '⭐⭐⭐' if weight > 30 else ('⭐⭐' if weight > 20 else '⭐')
    print(f'{dim:6s} {avg_wr:6.0f}% {bwr:7.0f}% {buwr:7.0f}% {freq:6.0f}% {stars:>10s}')

print('\n→ VOL(放量)最稀缺但质量最高，MA最常见但最不可靠。')

---
## 总结

### 五条边界条件

| # | 边界条件 | 结论 |
|---|---------|------|
| ① | 维度权重不等 | RSI+BB > MA+KLINE，放量权重应×2 |
| ② | 市场环境阈值 | 过去20日+5~+10%（强牛）= 最佳入场，73%胜率 |
| ③ | 死叉退出有毒 | 死叉退出6笔全亏，应取消 |
| ④ | 单维度质量 | VOL(72%牛市胜率) >> MA(53%) |
| ⑤ | 牛市用MACD | 牛市中MACD金叉裸奔(累计+22.84%)碾压共振(+0.36%) |

### V3 改进方向

```
牛市(MA20>MA60占比>50%): MACD金叉 + 放量 = 买入
震荡(30~50%):           共振≥3 + 放量 = 买入
熊市(<30%):            共振≥3 + RSI必须在30-50 + 放量 = 买入
暴跌(过去20日<-10%):    不交易

卖出: 10天到期 / -5%止损（取消死叉退出）
仓位: 牛市40% / 震荡30% / 熊市20%
```